# Set up in cloud

### For Colab notebooks, start here

In [ ]:
!git clone https://${GITHUB_TOKEN}@github.com/getsentry/grouping-trainer.git

In [ ]:
%cd grouping-trainer

### For Workbench notebooks, start here

In [ ]:
!git rev-parse --short HEAD

After running this `pip install` cell, restart the notebook session. TODO: activate venv instead

In [ ]:
!pip install -e .

In [ ]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

In [ ]:
!gsutil -m cp -r gs://grouping-data/runs/./2025-12-19-15-53-06-gte-output/training gte-finetuned/

In [ ]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://grouping-data/final_csvs .

# Run

In [ ]:
import os
import json
from datetime import datetime

import polars as pl
from pydantic import BaseModel
from sentence_transformers.util import pairwise_cos_sim
from tqdm.auto import tqdm

import grouping_trainer as gt
import utils

In [ ]:
class ModelConfig(BaseModel):
    name: str
    path: str
    truncate_dim: int | None = None
    batch_size: int = 1


class ModelConfigs(BaseModel):
    model_configs: list[ModelConfig]


class DataConfig(BaseModel):
    val_df_path: str
    sample_size: int | None = None

In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

In [ ]:
RUN_SHORTNAME = "test"
DATA_CONFIG = DataConfig(
    val_df_path="final_csvs/test.csv",
    sample_size=None,
)
MODEL_CONFIGS = ModelConfigs(
    model_configs=[
        ModelConfig(
            name="prod",
            path="/Users/kdubey/projects/seer/models/issue_grouping_v1/embeddings",
            truncate_dim=None,
        ),
        ModelConfig(
            name="gte-finetuned",
            path="gte-finetuned/training",
            truncate_dim=64,
        ),
    ]
)

OUTPUT_DIR = f"./{timestamp}-{RUN_SHORTNAME}"

In [ ]:
df = utils.load_val_df(path=DATA_CONFIG.val_df_path, sample_size=DATA_CONFIG.sample_size)

for model_config in tqdm(MODEL_CONFIGS.model_configs, desc="Processing models"):
    print(model_config)
    model = gt.utils.SentenceTransformer(
        model_config.path,
        trust_remote_code=True,
        truncate_dim=model_config.truncate_dim,
    )
    query_embeddings = model.encode(
        df["query_stacktrace_string"].to_list(),
        batch_size=model_config.batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    candidate_embeddings = model.encode(
        df["candidate_stacktrace_string"].to_list(),
        batch_size=model_config.batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    cos_sims = pairwise_cos_sim(query_embeddings, candidate_embeddings).detach().cpu().numpy()
    df = df.with_columns(pl.Series(name=f"cos_sim_{model_config.name}", values=cos_sims))
    print()

In [ ]:
print(df.columns)

# Upload

In [ ]:
os.mkdir(OUTPUT_DIR)

In [ ]:
with open(f"{OUTPUT_DIR}/model_configs.json", "w") as f:
    json.dump(MODEL_CONFIGS.model_dump(), f, indent=4)

with open(f"{OUTPUT_DIR}/data_config.json", "w") as f:
    json.dump(DATA_CONFIG.model_dump(), f, indent=4)

In [ ]:
df.write_csv(f"{OUTPUT_DIR}/test.csv")

In [ ]:
!gsutil -m rsync -r {OUTPUT_DIR} gs://grouping-data/similarities/{OUTPUT_DIR}

In [ ]:
!gsutil -m cp -r compare.ipynb gs://grouping-data/similarities/{OUTPUT_DIR}